In [ ]:
import random
from datasets import load_dataset

# =============================================================================
# 🌟 [튜터의 한마디] 🌟
# 안녕하세요, 코딩 탐험가님! 저는 AI 실습을 도와줄 친절하고 위트 있는 튜터입니다.
# 오늘 우리가 만날 데이터셋은 'nayohan/gpt4o-coding-eval-by-gemini1_5flash-ko'입니다.
# 🤖 이 데이터셋은 최신 AI 모델(GPT-4o 등)들이 코딩 문제를 얼마나 잘 풀었는지,
# 그리고 그 결과물(답변)이 얼마나 정확했는지 점수까지 매겨 놓은 'AI 성적표' 같은 곳이에요.
# 🧐 우리가 이 코딩 실습을 통해 데이터를 분석하면서, AI가 어떤 종류의 문제에 강하고 약한지
# 파헤쳐 볼 거예요! 기대되시죠? 파이썬 실력을 한 단계 업그레이드 해봅시다! 💪
# =============================================================================

# --- 설정 변수 ---
DATASET_NAME = "nayohan/gpt4o-coding-eval-by-gemini1_5flash-ko"
SPLIT_NAME = "train"
SAMPLE_COUNT = 10 # 실습의 속도를 위해 상위 10개 샘플만 사용합니다.

print(f"📚 로드할 데이터셋: {DATASET_NAME} ({SPLIT_NAME} 스플릿)")

# --- 데이터셋 로드 로직 (Streaming 우선 시도) ---
try:
    # 🚀 튜터 Tip: 스트리밍 모드는 데이터셋 전체를 메모리에 올리지 않아 매우 빠르고 효율적입니다!
    dataset = load_dataset(DATASET_NAME, split=SPLIT_NAME, streaming=True)
    print("✅ 스트리밍 모드(Streaming)로 데이터셋 로드 성공! 매우 빠르네요!")
    data_load_success = True
except Exception as e:
    # 🚨 만약 스트리밍이 실패하거나 네트워크 문제 등으로 멈춘다면,
    # 일반 로딩 방식으로 전환하여 최소한의 데이터라도 확보합니다.
    print(f"⚠️ 스트리밍 로드 실패 또는 오류 발생: {e}")
    print("💾 일반 로딩(Non-streaming) 모드로 전환하여 진행합니다...")
    try:
        dataset = load_dataset(DATASET_NAME, split=SPLIT_NAME)
        print("✅ 일반 로딩 모드로 데이터셋 로드 성공!")
        data_load_success = False
    except Exception as e_fallback:
        print(f"❌ 모든 로드 시도 실패! 데이터를 불러올 수 없습니다. 오류: {e_fallback}")
        exit()

# --- 데이터 샘플 확보 (스트리밍/비-스트리밍 공통 패턴 적용) ---
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)입니다.
    print(f"\n🔍 상위 {SAMPLE_COUNT}개 샘플을 가져옵니다...")
    sample_dataset_iterator = iter(dataset.take(SAMPLE_COUNT))
    
    # 스트리밍 데이터셋은 리스트로 변환하여 사용합니다.
    try:
        sample_data_list = [next(sample_dataset_iterator) for _ in range(SAMPLE_COUNT)]
    except StopIteration:
        sample_data_list = []
        print("ℹ️ 주의: 샘플 개수가 너무 적어 리스트 생성이 중단되었습니다.")
else:
    # 일반 데이터셋 (Dataset)인 경우 리스트로 변환합니다.
    sample_data_list = list(dataset.take(SAMPLE_COUNT))


# =============================================================================
# 💻 실습 1: 데이터 구조 파헤치기 (탐색적 분석)
# =============================================================================
print("\n" + "="*60)
print("📝 [실습 1] 데이터의 구조를 탐색하고, 핵심 정보를 뽑아내기")
print("="*60)

if not sample_data_list:
    print("🚨 샘플 데이터를 확보하지 못하여 실습을 진행할 수 없습니다.")
else:
    # 첫 번째 샘플을 통해 데이터 구조를 확인합니다.
    sample = sample_data_list[0]
    print(f"✨ 첫 번째 샘플의 'Instructions' (문제 내용):")
    print(f"   -> {sample['instructions'][:50]}... (내용 확인)")
    print("-" * 30)

    # 🎯 핵심 목표: 모델 성능 지표(스코어)와 문제 내용, 답변을 연관시키기.
    print("🔑 [핵심 분석 지표 확인]")
    print(f"  - 유사도 점수 (similarity_scores): {sample['similarity_scores']:.4f}")
    print(f"  - 정밀도 점수 (precision_scores): {sample['precision_scores']:.4f}")
    print(f"  - 목표 답변 (target_responses): {sample['target_responses'][:30]}...")


# =============================================================================
# ⚙️ 실습 2: 정량적 점수 기반 데이터 필터링 (데이터 분석)
# =============================================================================
print("\n" + "="*60)
print("📊 [실습 2] 성능 점수를 기준으로 '우수 샘플'만 골라내기")
print("="*60)

# 💡 가설: 유사도 점수와 정밀도 점수가 모두 높은 샘플은 '만점 케이스'일 가능성이 높다!
# 우리는 임계값(Threshold)을 설정하여 우수 샘플을 필터링 해보겠습니다.
SCORE_THRESHOLD = 0.85
excellent_samples = []

for i, sample in enumerate(sample_data_list):
    sim_score = sample['similarity_scores']
    pre_score = sample['precision_scores']

    # ✅ 조건 검사: 두 점수가 모두 임계값을 초과하는지 확인
    if sim_score >= SCORE_THRESHOLD and pre_score >= SCORE_THRESHOLD:
        excellent_samples.append(sample)

# 📝 결과 출력
print(f"✨ [분석 결과] 임계값 ({SCORE_THRESHOLD}) 이상의 우수 샘플 개수: {len(excellent_samples)}개")

if excellent_samples:
    print("🏆 우수 샘플 1개의 예시 (최상급 성능 사례):")
    best_sample = excellent_samples[0]
    print(f"  - Prompt: {best_sample['instructions'][:40]}...")
    print(f"  - Model ID: {best_sample['model_id']}")
    print(f"  - Scores: Sim={best_sample['similarity_scores']:.4f}, Pre={best_sample['precision_scores']:.4f}")
    print(f"  - Candidate Response: {best_sample['candidate_responses'][:50]}...")
else:
    print("😔 현재 샘플 내에서는 '최상급 성능'으로 분류할만한 샘플을 찾지 못했습니다. (임계값을 낮춰보세요!)")


# =============================================================================
# 🧠 실습 3: 비즈니스 로직 적용 (간단한 LLM 프롬프트 설계 시뮬레이션)
# =============================================================================
print("\n" + "="*60)
print("🤖 [실습 3] '실패 탐지' 로직을 만들어보기 (코드 검토 튜터 역할)")
print("="*60)

# 🛠️ 목표: 모델의 답변(candidate_responses)이 너무 짧거나,
#       혹은 특정 키워드가 빠진 경우를 '미흡'하다고 판단하는 함수를 만들어 봅시다.

def check_response_quality(sample):
    """
    답변의 길이와 내용에 기반하여 품질 점수를 임의로 매겨봅니다.
    (이것이 바로 '판단하는' AI의 기초 로직입니다!)
    """
    candidate = sample['candidate_responses']
    instructions = sample['instructions']
    
    # 1. 길이 체크: 답변이 50자 미만이면 일단 '경고'를 발생시킵니다.
    if len(candidate.strip()) < 50:
        return "⚠️ (Warning: Short) 답변이 너무 짧습니다. 디테일을 추가해주세요.", 0.7
    
    # 2. 필수 키워드 체크: 문제에 '함수'가 필요하다고 명시되어 있는데, 답변에 'def'가 없으면 감점!
    if "함수" in instructions and "def" not in candidate:
        return "❌ (Critical: Missing Def) 답변이 함수 형태로 보이지 않습니다. 코드 구조를 확인하세요.", 0.5
        
    # 3. 일반적 합격: 모든 체크를 통과한 경우
    return "✅ (Pass: OK) 답변 구조가 적절하며, 충분한 설명을 담고 있습니다.", 1.0

print("🌟 '응답 품질 검사기'를 실행합니다...")
fail_count = 0
successful_checks = 0

# 🛠️ 샘플 데이터에 로직을 적용합니다.
for i, sample in enumerate(sample_data_list):
    status, quality_score = check_response_quality(sample)
    
    # 🖍️ 재미를 위해 샘플마다 검사를 수행
    print(f"\n--- [Sample {i+1}] ---")
    print(f"🔍 Instructions (문제): {sample['instructions'][:40]}...")
    print(f"🤖 Candidate Response: {sample['candidate_responses'][:40]}...")
    print(f"✨ 품질 검사 결과: {status} (점수: {quality_score})")
    
    if quality_score < 0.8:
        fail_count += 1
    else:
        successful_checks += 1

print("\n================================================================")
print("💡 튜터의 결론:")
print(f"총 {len(sample_data_list)}개의 샘플을 분석했습니다.")
print(f"✨ 성공적인 답변 (점수 >= 0.8) 비율: {successful_checks / len(sample_data_list) * 100:.1f}%")
print("👏 축하합니다! 데이터셋의 구조와 통계적 로직을 사용하여, AI의 성과를 체계적으로 분석하는 능력을 보여주셨습니다. 정말 훌륭해요!")
print("궁금한 점이 있다면 언제든지 다시 질문해주세요. 파이썬 탐험은 재미있습니다!")
print("================================================================")